# Data Quality-Aware Cardiovascular Risk Modeling

This notebook is the first Kaggle-facing draft for the Byte2Beat project. It audits the provided tabular cardiovascular datasets, applies transparent clinical plausibility checks, trains interpretable and tree-based baselines, and prepares the core figures/tables for the final writeup.

**Research framing:** in cardiovascular ML, data quality, calibration, and subgroup reliability matter as much as raw model score.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))

TABLES = ROOT / 'outputs' / 'tables'
FIGURES = ROOT / 'outputs' / 'figures'
SEED = 42
print(ROOT)

## Reproducible Pipeline

The project scripts generate audit outputs, EDA tables, figures, baseline metrics, model comparison, calibration bins, and subgroup metrics.

In [ ]:
RUN_PIPELINE = True

if RUN_PIPELINE:
    scripts = [
        'scripts/profile_data.py',
        'scripts/eda_tabular.py',
        'scripts/generate_eda_figures.py',
        'scripts/baseline_tabular_numpy.py',
        'scripts/model_comparison.py',
        'scripts/cleaning_sensitivity.py',
    ]
    for script in scripts:
        print(f'Running {script}')
        subprocess.run([sys.executable, str(ROOT / script)], check=True)

## Cleaning Impact

The primary cardiac dataset is nearly balanced but contains implausible blood pressure and body-measurement values. We use transparent plausibility rules instead of silently trusting the preprocessed file.

In [ ]:
pd.read_csv(TABLES / 'cardio_cleaning_impact.csv')

In [ ]:
display(Image(filename=str(FIGURES / 'cardio_cleaning_flow.png')))
display(Image(filename=str(FIGURES / 'cardio_systolic_bp_raw_vs_clean.png')))

## EDA Signal

After plausibility cleaning, systolic blood pressure bands show a strong monotonic relationship with the observed cardio-positive rate.

In [ ]:
pd.read_csv(TABLES / 'cardio_target_rate_by_systolic_band.csv')

In [ ]:
display(Image(filename=str(FIGURES / 'cardio_target_by_bp_band.png')))
display(Image(filename=str(FIGURES / 'cardio_target_by_age_band.png')))

## Cleaning Sensitivity

A single cleaning rule can look arbitrary. We therefore compare four profiles: raw, lenient, current, and strict. The goal is to test whether the model result depends on one exact threshold set.

In [ ]:
pd.read_csv(TABLES / 'cleaning_sensitivity_summary.csv')

In [ ]:
sensitivity = pd.read_csv(TABLES / 'cleaning_sensitivity_metrics.csv')
sensitivity[['cleaning_profile', 'model', 'rows', 'auroc', 'auprc', 'accuracy', 'brier']].sort_values(['model', 'cleaning_profile'])

In [ ]:
display(Image(filename=str(FIGURES / 'cleaning_sensitivity_rows.png')))
display(Image(filename=str(FIGURES / 'cleaning_sensitivity_auroc.png')))
display(Image(filename=str(FIGURES / 'cleaning_sensitivity_brier.png')))

**Interpretation:** raw data weakens logistic regression, which is sensitive to implausible continuous values. Histogram gradient boosting remains close to AUROC 0.80 across profiles, and the current profile gives the strongest held-out AUROC/Brier combination. This supports the cleaning strategy without making the result depend on one fragile threshold choice.

## Model Comparison

We compare a dummy baseline, logistic regression, a shallow decision tree, random forest, and histogram gradient boosting. The headline result should not be model complexity alone; calibration and reliability remain central.

In [ ]:
model_comparison = pd.read_csv(TABLES / 'model_comparison.csv')
model_comparison[model_comparison['split'].eq('test')].sort_values(['dataset', 'auroc'], ascending=[True, False])

In [ ]:
pd.read_csv(TABLES / 'cross_validation_summary.csv').sort_values(['dataset', 'cv_auroc_mean'], ascending=[True, False])

In [ ]:
display(Image(filename=str(FIGURES / 'model_comparison_auroc.png')))
display(Image(filename=str(FIGURES / 'roc_curves.png')))
display(Image(filename=str(FIGURES / 'pr_curves.png')))

## Calibration and Subgroup Reliability

A cardiovascular risk model should be evaluated as a risk estimator, not just as a ranking function. We inspect calibration and subgroup metrics for the selected cardiac model.

In [ ]:
pd.read_csv(TABLES / 'cardio_clean_calibration_bins.csv')

In [ ]:
display(Image(filename=str(FIGURES / 'calibration_curve.png')))
subgroups = pd.read_csv(TABLES / 'subgroup_metrics.csv')
subgroups[['group_col', 'group_value', 'rows', 'positive_rate', 'auroc', 'sensitivity', 'specificity', 'brier']].head(20)

## Interpretability

The first logistic baseline gives clinically plausible directions: systolic blood pressure, age, cholesterol, and diastolic blood pressure are the largest standardized effects. The selected tree-based model is summarized with permutation importance.

In [ ]:
display(Image(filename=str(FIGURES / 'feature_importance.png')))
pd.read_csv(TABLES / 'feature_importance.csv').head(12)

## Current Conclusion

The cleaned cardiac dataset supports a stable risk-prediction baseline around AUROC 0.80. Histogram gradient boosting and random forest slightly outperform logistic regression, but the improvement is modest. The final project should therefore emphasize data-quality-aware modeling, calibration, subgroup reliability, and transparent limitations rather than only maximizing model score.

This notebook is not medical advice and is not a clinical decision tool.